In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)

In [2]:
BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

vehicles = pd.read_csv(RAW_DIR / "vehicles.csv")
telematics = pd.read_csv(RAW_DIR / "telematics.csv")
service_history = pd.read_csv(RAW_DIR / "service_history.csv")
support_tickets = pd.read_csv(RAW_DIR / "support_tickets.csv")
failure_labels = pd.read_csv(RAW_DIR / "failure_labels.csv")

telematics["date"] = pd.to_datetime(telematics["date"])
service_history["service_date"] = pd.to_datetime(service_history["service_date"])
support_tickets["ticket_date"] = pd.to_datetime(support_tickets["ticket_date"])

In [3]:
SNAPSHOT_DATE = telematics["date"].max()

SNAPSHOT_DATE

Timestamp('2026-04-30 00:00:00')

In [9]:
max_date = telematics["date"].max()
observation_start = telematics["date"].min()
observation_end = max_date

print("Observation window:", observation_start, "to", observation_end)

Observation window: 2026-01-01 00:00:00 to 2026-04-30 00:00:00


In [4]:
vehicle_features = vehicles.copy()

vehicle_features["is_electric"] = (vehicle_features["fuel_type"] == "Electric").astype(int)
vehicle_features["is_hybrid"] = (vehicle_features["fuel_type"] == "Hybrid").astype(int)
vehicle_features["is_diesel"] = (vehicle_features["fuel_type"] == "Diesel").astype(int)

vehicle_features.head()

,vehicle_id,customer_id,model,year,fuel_type,industry,mileage,vehicle_age,is_electric,is_hybrid,is_diesel
0,V00001,C0103,Transit,2018,Hybrid,Construction,125198,8,0,1,0
1,V00002,C0436,F-150,2020,Gasoline,Government,79155,6,0,0,0
2,V00003,C0349,Maverick,2020,Electric,Utilities,126449,6,1,0,0
3,V00004,C0271,F-150,2021,Diesel,Delivery,81700,5,0,0,1
4,V00005,C0107,Super Duty,2022,Hybrid,Utilities,11955,4,0,1,0


In [11]:
telematics.head()

,vehicle_id,date,engine_temp,oil_pressure,battery_voltage,brake_events,idle_hours,daily_miles
0,V00001,2026-01-01,93.350917,45.000877,12.134309,14,2.109864,69.096891
1,V00001,2026-01-02,97.449228,38.693877,12.657129,14,3.960258,31.999626
2,V00001,2026-01-03,99.705683,40.971823,12.256879,11,3.203803,66.622525
3,V00001,2026-01-04,96.828207,43.343437,12.079806,14,0.845895,71.984592
4,V00001,2026-01-05,94.173519,36.219813,12.477579,12,3.079657,104.067379


In [7]:
telematics_features = telematics.groupby("vehicle_id").agg(
    avg_engine_temp=("engine_temp", "mean"),
    max_engine_temp=("engine_temp", "max"),
    std_engine_temp=("engine_temp", "std"),

    avg_oil_pressure=("oil_pressure", "mean"),
    min_oil_pressure=("oil_pressure", "min"),
    std_oil_pressure=("oil_pressure", "std"),

    avg_battery_voltage=("battery_voltage", "mean"),
    min_battery_voltage=("battery_voltage", "min"),
    std_battery_voltage=("battery_voltage", "std"),

    total_brake_events=("brake_events", "sum"),
    avg_brake_events=("brake_events", "mean"),

    total_idle_hours=("idle_hours", "sum"),
    avg_idle_hours=("idle_hours", "mean"),

    total_miles_120d=("daily_miles", "sum"),
    avg_daily_miles=("daily_miles", "mean")
).reset_index()

telematics_features.head()

,vehicle_id,avg_engine_temp,max_engine_temp,std_engine_temp,avg_oil_pressure,min_oil_pressure,std_oil_pressure,avg_battery_voltage,min_battery_voltage,std_battery_voltage,total_brake_events,avg_brake_events,total_idle_hours,avg_idle_hours,total_miles_120d,avg_daily_miles
0,V00001,94.481981,106.062108,5.035404,37.074801,30.344797,3.044444,12.449984,11.677264,0.311851,1404,11.700000,330.840406,2.757003,9068.657408,75.572145
1,V00002,91.018243,107.257515,5.280720,37.661517,31.008005,2.889468,12.388175,11.593240,0.297356,1301,10.841667,302.018986,2.516825,8939.342723,74.494523
2,V00003,95.316620,109.442020,5.044226,42.522175,35.623743,2.966565,12.183761,11.465695,0.285913,1249,10.408333,299.390078,2.494917,9182.822235,76.523519
3,V00004,91.501110,103.218983,4.403340,39.306578,32.239884,2.703807,12.798029,12.110949,0.288667,1188,9.900000,310.080426,2.584004,8950.084504,74.584038
4,V00005,85.104323,100.018771,5.318115,41.404147,32.894843,3.127040,12.503618,11.908163,0.264745,1149,9.575000,302.138556,2.517821,8841.422327,73.678519


In [12]:
recent_30_start = max_date - pd.Timedelta(days=30)

telematics_30d = telematics[telematics["date"] >= recent_30_start]

telematics_30d_features = telematics_30d.groupby("vehicle_id").agg(
    avg_engine_temp_30d=("engine_temp", "mean"),
    max_engine_temp_30d=("engine_temp", "max"),

    avg_oil_pressure_30d=("oil_pressure", "mean"),
    min_oil_pressure_30d=("oil_pressure", "min"),

    avg_battery_voltage_30d=("battery_voltage", "mean"),
    min_battery_voltage_30d=("battery_voltage", "min"),

    brake_events_30d=("brake_events", "sum"),
    idle_hours_30d=("idle_hours", "sum"),
    miles_30d=("daily_miles", "sum")
).reset_index()

telematics_30d_features.head()

,vehicle_id,avg_engine_temp_30d,max_engine_temp_30d,avg_oil_pressure_30d,min_oil_pressure_30d,avg_battery_voltage_30d,min_battery_voltage_30d,brake_events_30d,idle_hours_30d,miles_30d
0,V00001,93.721165,104.068008,37.485937,32.538679,12.461453,11.754209,371,92.127523,2271.474392
1,V00002,91.612067,104.742948,37.211285,31.852468,12.353346,11.745808,329,75.872620,2496.837403
2,V00003,96.051931,109.442020,43.316361,35.623743,12.208740,11.675366,320,79.303012,2253.193631
3,V00004,92.625344,103.218983,39.246191,32.239884,12.822937,12.351553,323,77.691827,2141.198984
4,V00005,85.358585,94.737634,42.395639,37.561839,12.551600,12.022041,301,72.770769,2427.773147


In [5]:
def build_telematics_features(telematics_df, snapshot_date):
    df = telematics_df[telematics_df["date"] <= snapshot_date].copy()

    windows = {
        "7d": snapshot_date - pd.Timedelta(days=7),
        "30d": snapshot_date - pd.Timedelta(days=30),
        "90d": snapshot_date - pd.Timedelta(days=90)
    }

    features = []

    for window_name, start_date in windows.items():
        window_df = df[df["date"] > start_date]

        agg = window_df.groupby("vehicle_id").agg(
            **{
                f"avg_engine_temp_{window_name}": ("engine_temp", "mean"),
                f"max_engine_temp_{window_name}": ("engine_temp", "max"),
                f"std_engine_temp_{window_name}": ("engine_temp", "std"),

                f"avg_oil_pressure_{window_name}": ("oil_pressure", "mean"),
                f"min_oil_pressure_{window_name}": ("oil_pressure", "min"),
                f"std_oil_pressure_{window_name}": ("oil_pressure", "std"),

                f"avg_battery_voltage_{window_name}": ("battery_voltage", "mean"),
                f"min_battery_voltage_{window_name}": ("battery_voltage", "min"),
                f"std_battery_voltage_{window_name}": ("battery_voltage", "std"),

                f"total_brake_events_{window_name}": ("brake_events", "sum"),
                f"avg_brake_events_{window_name}": ("brake_events", "mean"),

                f"total_idle_hours_{window_name}": ("idle_hours", "sum"),
                f"avg_idle_hours_{window_name}": ("idle_hours", "mean"),

                f"total_miles_{window_name}": ("daily_miles", "sum"),
                f"avg_daily_miles_{window_name}": ("daily_miles", "mean"),
            }
        ).reset_index()

        features.append(agg)

    final = features[0]

    for feat in features[1:]:
        final = final.merge(feat, on="vehicle_id", how="outer")

    return final

In [6]:
telematics_features = build_telematics_features(telematics, SNAPSHOT_DATE)

telematics_features.head()

,vehicle_id,avg_engine_temp_7d,max_engine_temp_7d,std_engine_temp_7d,avg_oil_pressure_7d,min_oil_pressure_7d,std_oil_pressure_7d,avg_battery_voltage_7d,min_battery_voltage_7d,std_battery_voltage_7d,total_brake_events_7d,avg_brake_events_7d,total_idle_hours_7d,avg_idle_hours_7d,total_miles_7d,avg_daily_miles_7d,avg_engine_temp_30d,max_engine_temp_30d,std_engine_temp_30d,avg_oil_pressure_30d,min_oil_pressure_30d,std_oil_pressure_30d,avg_battery_voltage_30d,min_battery_voltage_30d,std_battery_voltage_30d,total_brake_events_30d,avg_brake_events_30d,total_idle_hours_30d,avg_idle_hours_30d,total_miles_30d,avg_daily_miles_30d,avg_engine_temp_90d,max_engine_temp_90d,std_engine_temp_90d,avg_oil_pressure_90d,min_oil_pressure_90d,std_oil_pressure_90d,avg_battery_voltage_90d,min_battery_voltage_90d,std_battery_voltage_90d,total_brake_events_90d,avg_brake_events_90d,total_idle_hours_90d,avg_idle_hours_90d,total_miles_90d,avg_daily_miles_90d
0,V00001,94.508207,99.295782,3.111666,38.333959,35.257880,2.578602,12.446781,11.754209,0.364661,69,9.857143,22.575540,3.225077,377.165862,53.880837,93.705715,104.068008,4.965239,37.567064,32.538679,2.838862,12.466617,11.754209,0.329119,358,11.933333,89.914897,2.997163,2183.301821,72.776727,94.160422,104.068008,4.865140,36.948813,30.344797,2.939115,12.446237,11.677264,0.318883,1060,11.777778,247.341126,2.748235,6735.265319,74.836281
1,V00002,90.358889,97.493847,4.186492,36.851012,31.852468,2.697149,12.339796,11.881754,0.286245,74,10.571429,16.656462,2.379495,646.697403,92.385343,91.175904,104.742948,5.184712,37.157510,31.852468,2.088869,12.351222,11.745808,0.336287,316,10.533333,72.085571,2.402852,2433.543191,81.118106,91.281600,107.257515,5.337957,37.369263,31.008005,2.831142,12.388240,11.593240,0.301596,939,10.433333,224.315548,2.492395,6889.390338,76.548782
2,V00003,94.689211,105.125238,6.224421,42.430697,35.623743,3.433602,12.191515,11.854908,0.329224,71,10.142857,19.174044,2.739149,447.780339,63.968620,96.124061,109.442020,5.350345,43.360339,35.623743,2.703251,12.193368,11.675366,0.292528,308,10.266667,76.708020,2.556934,2150.078799,71.669293,95.701266,109.442020,5.208004,42.616438,35.623743,3.044707,12.197484,11.619928,0.283439,949,10.544444,225.877864,2.509754,6640.195799,73.779953
3,V00004,94.157495,98.201911,4.318486,41.204531,36.834020,2.912992,12.802232,12.567930,0.232893,68,9.714286,19.427963,2.775423,525.556957,75.079565,92.698486,103.218983,4.529911,39.207338,32.239884,3.083607,12.812283,12.351553,0.265818,308,10.266667,75.400369,2.513346,2050.972601,68.365753,91.509367,103.218983,4.499426,39.395798,32.239884,2.830771,12.797772,12.110949,0.282588,910,10.111111,232.665500,2.585172,6684.736225,74.274847
4,V00005,84.362243,88.248591,2.957351,43.775780,40.044047,2.081022,12.569562,12.022041,0.296370,59,8.428571,16.565031,2.366433,590.490661,84.355809,85.585079,94.737634,5.803693,42.454797,37.561839,2.826356,12.547206,12.022041,0.229444,290,9.666667,69.623333,2.320778,2341.083286,78.036110,85.171856,100.018771,5.637709,41.578059,34.277760,3.176793,12.532570,11.908163,0.273031,866,9.622222,228.265791,2.536287,6623.169474,73.590772


In [13]:
early_period = telematics[telematics["date"] < recent_30_start]
recent_period = telematics[telematics["date"] >= recent_30_start]

early_features = early_period.groupby("vehicle_id").agg(
    avg_engine_temp_early=("engine_temp", "mean"),
    avg_oil_pressure_early=("oil_pressure", "mean"),
    avg_battery_voltage_early=("battery_voltage", "mean")
).reset_index()

recent_features = recent_period.groupby("vehicle_id").agg(
    avg_engine_temp_recent=("engine_temp", "mean"),
    avg_oil_pressure_recent=("oil_pressure", "mean"),
    avg_battery_voltage_recent=("battery_voltage", "mean")
).reset_index()

trend_features = early_features.merge(recent_features, on="vehicle_id", how="outer")

trend_features["engine_temp_trend"] = (
    trend_features["avg_engine_temp_recent"] - trend_features["avg_engine_temp_early"]
)

trend_features["oil_pressure_trend"] = (
    trend_features["avg_oil_pressure_recent"] - trend_features["avg_oil_pressure_early"]
)

trend_features["battery_voltage_trend"] = (
    trend_features["avg_battery_voltage_recent"] - trend_features["avg_battery_voltage_early"]
)

trend_features = trend_features[[
    "vehicle_id",
    "engine_temp_trend",
    "oil_pressure_trend",
    "battery_voltage_trend"
]]

trend_features.head()

,vehicle_id,engine_temp_trend,oil_pressure_trend,battery_voltage_trend
0,V00001,-1.025819,0.554340,0.015465
1,V00002,0.800662,-0.607055,-0.046961
2,V00003,0.991431,1.070813,0.033679
3,V00004,1.515820,-0.081420,0.033584
4,V00005,0.342825,1.336843,0.064695


In [15]:
service_features = service_history.groupby("vehicle_id").agg(
    repair_count=("repair_type", "count"),
    total_repair_cost=("repair_cost", "sum"),
    avg_repair_cost=("repair_cost", "mean"),
    last_service_date=("service_date", "max")
).reset_index()

service_features["days_since_last_service"] = (
    max_date - service_features["last_service_date"]
).dt.days

service_features = service_features.drop(columns=["last_service_date"])

service_features.head()

,vehicle_id,repair_count,total_repair_cost,avg_repair_cost,days_since_last_service
0,V00001,2,371.98,185.9900,69
1,V00002,4,2547.91,636.9775,11
2,V00003,3,697.47,232.4900,72
3,V00004,1,548.91,548.9100,8
4,V00005,5,3909.95,781.9900,0


In [16]:
repair_type_features = (
    pd.get_dummies(service_history[["vehicle_id", "repair_type"]], columns=["repair_type"])
    .groupby("vehicle_id")
    .sum()
    .reset_index()
)

repair_type_features.head()

,vehicle_id,repair_type_Battery Replacement,repair_type_Brake Repair,repair_type_Cooling System,repair_type_Engine Inspection,repair_type_Oil Change,repair_type_Transmission Repair
0,V00001,1,0,0,0,1,0
1,V00002,0,1,0,0,2,1
2,V00003,0,1,0,0,2,0
3,V00004,0,1,0,0,0,0
4,V00005,2,0,0,2,0,1


In [17]:
ticket_features = support_tickets.groupby("vehicle_id").agg(
    ticket_count=("ticket_id", "count"),
    high_priority_tickets=("priority", lambda x: (x == "High").sum()),
    medium_priority_tickets=("priority", lambda x: (x == "Medium").sum()),
    low_priority_tickets=("priority", lambda x: (x == "Low").sum())
).reset_index()

ticket_features.head()

,vehicle_id,ticket_count,high_priority_tickets,medium_priority_tickets,low_priority_tickets
0,V00001,1,1,0,0
1,V00005,1,0,1,0
2,V00007,3,1,1,1
3,V00008,3,1,0,2
4,V00009,1,0,1,0


In [18]:
ticket_features = support_tickets.groupby("vehicle_id").agg(
    ticket_count=("ticket_id", "count"),
    high_priority_tickets=("priority", lambda x: (x == "High").sum()),
    medium_priority_tickets=("priority", lambda x: (x == "Medium").sum()),
    low_priority_tickets=("priority", lambda x: (x == "Low").sum())
).reset_index()

ticket_features.head()

,vehicle_id,ticket_count,high_priority_tickets,medium_priority_tickets,low_priority_tickets
0,V00001,1,1,0,0
1,V00005,1,0,1,0
2,V00007,3,1,1,1
3,V00008,3,1,0,2
4,V00009,1,0,1,0


In [20]:
ticket_text = support_tickets.groupby("vehicle_id")["complaint_text"].apply(
    lambda x: " ".join(x).lower()
).reset_index()

keywords = {
    "complaint_overheating": "overheating",
    "complaint_battery": "battery",
    "complaint_brake": "brake",
    "complaint_transmission": "transmission",
    "complaint_oil_pressure": "oil pressure",
    "complaint_check_engine": "check engine",
    "complaint_acceleration": "acceleration"
}

for feature_name, keyword in keywords.items():
    ticket_text[feature_name] = ticket_text["complaint_text"].str.contains(keyword).astype(int)

ticket_text_features = ticket_text.drop(columns=["complaint_text"])

ticket_text_features.head()

,vehicle_id,complaint_overheating,complaint_battery,complaint_brake,complaint_transmission,complaint_oil_pressure,complaint_check_engine,complaint_acceleration
0,V00001,0,0,0,0,1,0,0
1,V00005,0,0,0,0,0,0,0
2,V00007,0,0,0,0,1,0,1
3,V00008,1,0,0,0,0,0,0
4,V00009,0,0,0,0,0,0,0


In [21]:
training_df = vehicle_features.merge(telematics_features, on="vehicle_id", how="left")
training_df = training_df.merge(telematics_30d_features, on="vehicle_id", how="left")
training_df = training_df.merge(trend_features, on="vehicle_id", how="left")
training_df = training_df.merge(service_features, on="vehicle_id", how="left")
training_df = training_df.merge(repair_type_features, on="vehicle_id", how="left")
training_df = training_df.merge(ticket_features, on="vehicle_id", how="left")
training_df = training_df.merge(ticket_text_features, on="vehicle_id", how="left")
training_df = training_df.merge(failure_labels[["vehicle_id", "failure_next_30_days"]], on="vehicle_id", how="left")

training_df.head()

,vehicle_id,customer_id,model,year,fuel_type,industry,mileage,vehicle_age,is_electric,is_hybrid,is_diesel,avg_engine_temp,max_engine_temp,std_engine_temp,avg_oil_pressure,min_oil_pressure,std_oil_pressure,avg_battery_voltage,min_battery_voltage,std_battery_voltage,total_brake_events,avg_brake_events,total_idle_hours,avg_idle_hours,total_miles_120d,avg_daily_miles,avg_engine_temp_30d,max_engine_temp_30d,avg_oil_pressure_30d,min_oil_pressure_30d,avg_battery_voltage_30d,min_battery_voltage_30d,brake_events_30d,idle_hours_30d,miles_30d,engine_temp_trend,oil_pressure_trend,battery_voltage_trend,repair_count,total_repair_cost,avg_repair_cost,days_since_last_service,repair_type_Battery Replacement,repair_type_Brake Repair,repair_type_Cooling System,repair_type_Engine Inspection,repair_type_Oil Change,repair_type_Transmission Repair,ticket_count,high_priority_tickets,medium_priority_tickets,low_priority_tickets,complaint_overheating,complaint_battery,complaint_brake,complaint_transmission,complaint_oil_pressure,complaint_check_engine,complaint_acceleration,failure_next_30_days
0,V00001,C0103,Transit,2018,Hybrid,Construction,125198,8,0,1,0,94.481981,106.062108,5.035404,37.074801,30.344797,3.044444,12.449984,11.677264,0.311851,1404,11.700000,330.840406,2.757003,9068.657408,75.572145,93.721165,104.068008,37.485937,32.538679,12.461453,11.754209,371,92.127523,2271.474392,-1.025819,0.554340,0.015465,2.0,371.98,185.9900,69.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1
1,V00002,C0436,F-150,2020,Gasoline,Government,79155,6,0,0,0,91.018243,107.257515,5.280720,37.661517,31.008005,2.889468,12.388175,11.593240,0.297356,1301,10.841667,302.018986,2.516825,8939.342723,74.494523,91.612067,104.742948,37.211285,31.852468,12.353346,11.745808,329,75.872620,2496.837403,0.800662,-0.607055,-0.046961,4.0,2547.91,636.9775,11.0,0.0,1.0,0.0,0.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,V00003,C0349,Maverick,2020,Electric,Utilities,126449,6,1,0,0,95.316620,109.442020,5.044226,42.522175,35.623743,2.966565,12.183761,11.465695,0.285913,1249,10.408333,299.390078,2.494917,9182.822235,76.523519,96.051931,109.442020,43.316361,35.623743,12.208740,11.675366,320,79.303012,2253.193631,0.991431,1.070813,0.033679,3.0,697.47,232.4900,72.0,0.0,1.0,0.0,0.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,V00004,C0271,F-150,2021,Diesel,Delivery,81700,5,0,0,1,91.501110,103.218983,4.403340,39.306578,32.239884,2.703807,12.798029,12.110949,0.288667,1188,9.900000,310.080426,2.584004,8950.084504,74.584038,92.625344,103.218983,39.246191,32.239884,12.822937,12.351553,323,77.691827,2141.198984,1.515820,-0.081420,0.033584,1.0,548.91,548.9100,8.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,V00005,C0107,Super Duty,2022,Hybrid,Utilities,11955,4,0,1,0,85.104323,100.018771,5.318115,41.404147,32.894843,3.127040,12.503618,11.908163,0.264745,1149,9.575000,302.138556,2.517821,8841.422327,73.678519,85.358585,94.737634,42.395639,37.561839,12.551600,12.022041,301,72.770769,2427.773147,0.342825,1.336843,0.064695,5.0,3909.95,781.9900,0.0,2.0,0.0,0.0,2.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [22]:
numeric_cols = training_df.select_dtypes(include=["number"]).columns

training_df[numeric_cols] = training_df[numeric_cols].fillna(0)

training_df.isna().sum().sort_values(ascending=False).head(20)

vehicle_id                         0
customer_id                        0
brake_events_30d                   0
idle_hours_30d                     0
miles_30d                          0
engine_temp_trend                  0
oil_pressure_trend                 0
battery_voltage_trend              0
repair_count                       0
total_repair_cost                  0
avg_repair_cost                    0
days_since_last_service            0
repair_type_Battery Replacement    0
repair_type_Brake Repair           0
repair_type_Cooling System         0
repair_type_Engine Inspection      0
repair_type_Oil Change             0
repair_type_Transmission Repair    0
ticket_count                       0
high_priority_tickets              0
dtype: int64

In [23]:
categorical_cols = ["model", "fuel_type", "industry"]

training_df_encoded = pd.get_dummies(
    training_df,
    columns=categorical_cols,
    drop_first=True
)

training_df_encoded.head()

,vehicle_id,customer_id,year,mileage,vehicle_age,is_electric,is_hybrid,is_diesel,avg_engine_temp,max_engine_temp,std_engine_temp,avg_oil_pressure,min_oil_pressure,std_oil_pressure,avg_battery_voltage,min_battery_voltage,std_battery_voltage,total_brake_events,avg_brake_events,total_idle_hours,avg_idle_hours,total_miles_120d,avg_daily_miles,avg_engine_temp_30d,max_engine_temp_30d,avg_oil_pressure_30d,min_oil_pressure_30d,avg_battery_voltage_30d,min_battery_voltage_30d,brake_events_30d,idle_hours_30d,miles_30d,engine_temp_trend,oil_pressure_trend,battery_voltage_trend,repair_count,total_repair_cost,avg_repair_cost,days_since_last_service,repair_type_Battery Replacement,repair_type_Brake Repair,repair_type_Cooling System,repair_type_Engine Inspection,repair_type_Oil Change,repair_type_Transmission Repair,ticket_count,high_priority_tickets,medium_priority_tickets,low_priority_tickets,complaint_overheating,complaint_battery,complaint_brake,complaint_transmission,complaint_oil_pressure,complaint_check_engine,complaint_acceleration,failure_next_30_days,model_F-150,model_Maverick,model_Super Duty,model_Transit,fuel_type_Electric,fuel_type_Gasoline,fuel_type_Hybrid,industry_Delivery,industry_Government,industry_Logistics,industry_Utilities
0,V00001,C0103,2018,125198,8,0,1,0,94.481981,106.062108,5.035404,37.074801,30.344797,3.044444,12.449984,11.677264,0.311851,1404,11.700000,330.840406,2.757003,9068.657408,75.572145,93.721165,104.068008,37.485937,32.538679,12.461453,11.754209,371,92.127523,2271.474392,-1.025819,0.554340,0.015465,2.0,371.98,185.9900,69.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1,False,False,False,True,False,False,True,False,False,False,False
1,V00002,C0436,2020,79155,6,0,0,0,91.018243,107.257515,5.280720,37.661517,31.008005,2.889468,12.388175,11.593240,0.297356,1301,10.841667,302.018986,2.516825,8939.342723,74.494523,91.612067,104.742948,37.211285,31.852468,12.353346,11.745808,329,75.872620,2496.837403,0.800662,-0.607055,-0.046961,4.0,2547.91,636.9775,11.0,0.0,1.0,0.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,True,False,False,False,False,True,False,False,True,False,False
2,V00003,C0349,2020,126449,6,1,0,0,95.316620,109.442020,5.044226,42.522175,35.623743,2.966565,12.183761,11.465695,0.285913,1249,10.408333,299.390078,2.494917,9182.822235,76.523519,96.051931,109.442020,43.316361,35.623743,12.208740,11.675366,320,79.303012,2253.193631,0.991431,1.070813,0.033679,3.0,697.47,232.4900,72.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,False,True,False,False,True,False,False,False,False,False,True
3,V00004,C0271,2021,81700,5,0,0,1,91.501110,103.218983,4.403340,39.306578,32.239884,2.703807,12.798029,12.110949,0.288667,1188,9.900000,310.080426,2.584004,8950.084504,74.584038,92.625344,103.218983,39.246191,32.239884,12.822937,12.351553,323,77.691827,2141.198984,1.515820,-0.081420,0.033584,1.0,548.91,548.9100,8.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,True,False,False,False,False,False,False,True,False,False,False
4,V00005,C0107,2022,11955,4,0,1,0,85.104323,100.018771,5.318115,41.404147,32.894843,3.127040,12.503618,11.908163,0.264745,1149,9.575000,302.138556,2.517821,8841.422327,73.678519,85.358585,94.737634,42.395639,37.561839,12.551600,12.022041,301,72.770769,2427.773147,0.342825,1.336843,0.064695,5.0,3909.95,781.9900,0.0,2.0,0.0,0.0,2.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,False,False,True,False,False,False,True,False,False,False,True


In [24]:
model_training_table = training_df_encoded.drop(columns=[
    "customer_id"
])

In [25]:
print("Final training table shape:", model_training_table.shape)
print("Target distribution:")
print(model_training_table["failure_next_30_days"].value_counts(normalize=True))

model_training_table.head()

Final training table shape: (3000, 67)
Target distribution:
failure_next_30_days
0    0.75
1    0.25
Name: proportion, dtype: float64


,vehicle_id,year,mileage,vehicle_age,is_electric,is_hybrid,is_diesel,avg_engine_temp,max_engine_temp,std_engine_temp,avg_oil_pressure,min_oil_pressure,std_oil_pressure,avg_battery_voltage,min_battery_voltage,std_battery_voltage,total_brake_events,avg_brake_events,total_idle_hours,avg_idle_hours,total_miles_120d,avg_daily_miles,avg_engine_temp_30d,max_engine_temp_30d,avg_oil_pressure_30d,min_oil_pressure_30d,avg_battery_voltage_30d,min_battery_voltage_30d,brake_events_30d,idle_hours_30d,miles_30d,engine_temp_trend,oil_pressure_trend,battery_voltage_trend,repair_count,total_repair_cost,avg_repair_cost,days_since_last_service,repair_type_Battery Replacement,repair_type_Brake Repair,repair_type_Cooling System,repair_type_Engine Inspection,repair_type_Oil Change,repair_type_Transmission Repair,ticket_count,high_priority_tickets,medium_priority_tickets,low_priority_tickets,complaint_overheating,complaint_battery,complaint_brake,complaint_transmission,complaint_oil_pressure,complaint_check_engine,complaint_acceleration,failure_next_30_days,model_F-150,model_Maverick,model_Super Duty,model_Transit,fuel_type_Electric,fuel_type_Gasoline,fuel_type_Hybrid,industry_Delivery,industry_Government,industry_Logistics,industry_Utilities
0,V00001,2018,125198,8,0,1,0,94.481981,106.062108,5.035404,37.074801,30.344797,3.044444,12.449984,11.677264,0.311851,1404,11.700000,330.840406,2.757003,9068.657408,75.572145,93.721165,104.068008,37.485937,32.538679,12.461453,11.754209,371,92.127523,2271.474392,-1.025819,0.554340,0.015465,2.0,371.98,185.9900,69.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1,False,False,False,True,False,False,True,False,False,False,False
1,V00002,2020,79155,6,0,0,0,91.018243,107.257515,5.280720,37.661517,31.008005,2.889468,12.388175,11.593240,0.297356,1301,10.841667,302.018986,2.516825,8939.342723,74.494523,91.612067,104.742948,37.211285,31.852468,12.353346,11.745808,329,75.872620,2496.837403,0.800662,-0.607055,-0.046961,4.0,2547.91,636.9775,11.0,0.0,1.0,0.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,True,False,False,False,False,True,False,False,True,False,False
2,V00003,2020,126449,6,1,0,0,95.316620,109.442020,5.044226,42.522175,35.623743,2.966565,12.183761,11.465695,0.285913,1249,10.408333,299.390078,2.494917,9182.822235,76.523519,96.051931,109.442020,43.316361,35.623743,12.208740,11.675366,320,79.303012,2253.193631,0.991431,1.070813,0.033679,3.0,697.47,232.4900,72.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,False,True,False,False,True,False,False,False,False,False,True
3,V00004,2021,81700,5,0,0,1,91.501110,103.218983,4.403340,39.306578,32.239884,2.703807,12.798029,12.110949,0.288667,1188,9.900000,310.080426,2.584004,8950.084504,74.584038,92.625344,103.218983,39.246191,32.239884,12.822937,12.351553,323,77.691827,2141.198984,1.515820,-0.081420,0.033584,1.0,548.91,548.9100,8.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,True,False,False,False,False,False,False,True,False,False,False
4,V00005,2022,11955,4,0,1,0,85.104323,100.018771,5.318115,41.404147,32.894843,3.127040,12.503618,11.908163,0.264745,1149,9.575000,302.138556,2.517821,8841.422327,73.678519,85.358585,94.737634,42.395639,37.561839,12.551600,12.022041,301,72.770769,2427.773147,0.342825,1.336843,0.064695,5.0,3909.95,781.9900,0.0,2.0,0.0,0.0,2.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,False,False,True,False,False,False,True,False,False,False,True


In [26]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

model_training_table.to_csv(PROCESSED_DIR / "model_training_table.csv", index=False)

print("Saved model training table:")
print(PROCESSED_DIR / "model_training_table.csv")

Saved model training table:
../data/processed/model_training_table.csv
